# 05 - Robust (degradation-aware) training and comparison
Runtime: **T4 GPU** for the training cells. The evaluation, robustness and comparison
cells also work on a CPU runtime later. Run the cells **one at a time**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata

token = userdata.get('GH_TOKEN')
REPO = 'ThemistoklisSakkas/sar-ship-robustness'
DEST = '/content/sar-ship-robustness'
if os.path.isdir(DEST):
    !git -C {DEST} pull -q https://{token}@github.com/{REPO}.git main
else:
    !git clone -q https://{token}@github.com/{REPO}.git {DEST}
    !git -C {DEST} remote set-url origin https://github.com/{REPO}.git
%cd {DEST}
!pip install -q -r requirements.txt

DRIVE_ROOT = '/content/drive/MyDrive/sar-ship-robustness'
DATA_ROOT = f'{DRIVE_ROOT}/data'
RUNS_ROOT = f'{DRIVE_ROOT}/runs'
NAME = 'robust_yolo11n'
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 1 - smoke test (about 1 min on GPU)
One epoch, to check that the degradation hook works before spending 30-60 min.
Look at the training batch images below: some chips must look blurred, speckled,
low-contrast or blocky (lower resolution).

In [ ]:
!python -m sarship.train --data-root {DATA_ROOT} --runs-root {RUNS_ROOT} --name smoke_robust --overwrite --set robust_train.enabled=true train.epochs=1

In [ ]:
from IPython.display import Image, display
for i in range(3):
    path = f'{RUNS_ROOT}/smoke_robust/train_batch{i}.jpg'
    if os.path.exists(path):
        display(Image(path, width=900))

## Step 2 - full robust training (30-60 min on GPU)
After a disconnect: re-run the two setup cells, then the resume cell.

In [ ]:
!python -m sarship.train --data-root {DATA_ROOT} --runs-root {RUNS_ROOT} --name {NAME} --set robust_train.enabled=true

In [ ]:
# Only after a disconnect (the saved config of the run is reused automatically):
# !python -m sarship.train --data-root {DATA_ROOT} --runs-root {RUNS_ROOT} --name {NAME} --resume

## Step 3 - evaluation, robustness study, comparison (CPU is fine)

In [ ]:
!python -m sarship.evaluate --weights {RUNS_ROOT}/{NAME}/weights/best.pt --data-root {DATA_ROOT} --runs-root {RUNS_ROOT} --name {NAME}

In [ ]:
!python -m sarship.robustness --weights {RUNS_ROOT}/{NAME}/weights/best.pt --data-root {DATA_ROOT} --runs-root {RUNS_ROOT} --name {NAME} --work-dir /content/degraded_robust

In [ ]:
!python -m sarship.compare --data-root {DATA_ROOT} --runs-root {RUNS_ROOT}

In [ ]:
for path in [f'{RUNS_ROOT}/{NAME}/evaluation/training_curves.png',
             f'{RUNS_ROOT}/{NAME}/comparison/comparison.png']:
    print(path.split('/')[-1])
    display(Image(path))